In [0]:
from pyspark.sql import functions as F

bronze_products = spark.table("brazilian_ecommerce.prod.bronze_products")

staging_products = (
    bronze_products
    .withColumn(
        "product_category_name",
        F.lower(F.regexp_replace(F.col("product_category_name"), "_", " "))
    )
    .withColumn("product_name_lenght", F.col("product_name_lenght").cast("int"))
    .withColumn("product_description_lenght", F.col("product_description_lenght").cast("int"))
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast("int"))
    .withColumn("product_weight_g", F.col("product_weight_g").cast("int"))
    .withColumn("product_length_cm", F.col("product_length_cm").cast("int"))
    .withColumn("product_height_cm", F.col("product_height_cm").cast("int"))
    .withColumn("product_width_cm", F.col("product_width_cm").cast("int"))
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
    .withColumn("_loaded_at", F.current_timestamp())
)

full_table_name = "brazilian_ecommerce.prod.staging_products"

(
    staging_products.write
    .mode("overwrite")
    .option("comment", "products staging table")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN product_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_staging_products PRIMARY KEY (product_id)
""")

print(f"Loaded {staging_products.count()} rows into {full_table_name}")